# Day06下午个人项目：电商用户数据可视化

姓名/学号或GitHub用户名：**请填写**  
第5天专题（A/B/C/D/E）：**请填写**

本Notebook需要完成4张独立图、1张综合图和1份图表清单。请阅读`docs/day06_student_visualization_manual.md`后开始。


## 项目规则

1. 使用第4天清洗数据，并核对第5天个人分析结果；
2. 柱状图和散点图必做；折线图只能用于时间或有序阶段；
3. 饼图只用于少量类别的整体构成，必要时改用柱状图；
4. 每张图写“观察—证据—边界”；
5. 输出文件名和目录不得修改，以便第7天Flask直接复用。


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

STUDENT_ID = "请填写学号或GitHub用户名"
TOPIC = "请填写A/B/C/D/E"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "PingFang SC",
    "Heiti SC", "Arial Unicode MS", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到第4天清洗数据，请先完成Day04。")


ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
DAY05_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR = ROOT / "output" / "day06_visualization"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR.relative_to(ROOT))


学生： 请填写学号或GitHub用户名
专题： 请填写A/B/C/D/E
输出： output\day06_visualization


## 检查点1：输入与业务问题

先验证4个输入文件，再写出4个问题。不要在尚未理解指标时直接绘图。


In [2]:
required_inputs = [
    DATA_PATH,
    DAY05_DIR / "overall_metrics.csv",
    DAY05_DIR / "segment_analysis.csv",
    DAY05_DIR / "cross_analysis.csv",
]
missing_inputs = [str(path.relative_to(ROOT)) for path in required_inputs if not path.exists()]
assert not missing_inputs, f"缺少输入文件：{missing_inputs}"

df = pd.read_csv(DATA_PATH)
overall_metrics = pd.read_csv(required_inputs[1])
segment_analysis = pd.read_csv(required_inputs[2])
cross_analysis = pd.read_csv(required_inputs[3])

assert df.shape[0] == 5630, f"清洗数据行数异常：{df.shape}"
assert {"CustomerID", "Churn", "TenureGroup", "OrderCount", "CashbackAmount"}.issubset(df.columns)
assert set(df["Churn"].dropna().unique()).issubset({0, 1})

display(overall_metrics)
display(segment_analysis.head())
display(cross_analysis.head())
print("检查点1A通过：输入文件有效")


AssertionError: 缺少输入文件：['output\\day05_analysis\\overall_metrics.csv', 'output\\day05_analysis\\segment_analysis.csv', 'output\\day05_analysis\\cross_analysis.csv']

In [3]:
# TODO：填写4个业务问题和图表选择理由
business_questions = {
    "category_bar": "不同城市等级的用户数量和流失率分别是多少？",
    "behavior_scatter": "用户订单数量与返现金额之间存在怎样的关系？流失用户与留存用户在行为上有何差异？",
    "ordered_line": "随着用户使用时长增加，用户流失率呈现怎样的变化趋势？",
    "composition_chart": "电商平台用户的首选支付方式分布情况如何？",
}

chart_reasons = {
    "category_bar": "柱状图适合对比不同类别（城市等级）之间的数值差异，能清晰展示各组用户数和流失率的高低对比。",
    "behavior_scatter": "散点图用于展示两个连续变量之间的关系，按流失状态分色可以直观观察两类用户的行为分布差异。",
    "ordered_line": "折线图适合展示具有明确顺序的变量（使用时长分组）随顺序变化的趋势，便于观察流失率随时长的变化规律。",
    "composition_chart": "环形图（饼图变种）适合展示构成比例关系，能清晰呈现各支付方式在用户总体中的占比分布。",
}

assert all(text.strip() for text in business_questions.values()), "请填写4个业务问题"
assert all(text.strip() for text in chart_reasons.values()), "请填写4个图表选择理由"
print("检查点1B通过：业务问题和选择理由已填写")

AssertionError: 请填写4个业务问题

## 任务1：类别比较柱状图

要求：选择一个离散分组字段，计算用户数和一个核心指标；若绘制比率，标签中必须同时给出样本量。


In [ ]:
# TODO：完成绘图数据。建议使用自己的第5天主分组字段。
category_field = "CityTier"
category_summary = (
    df.groupby(category_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)

assert category_field in df.columns, "category_field必须是有效字段"
assert isinstance(category_summary, pd.DataFrame), "category_summary必须是DataFrame"
assert {category_field, "用户数"}.issubset(category_summary.columns)
display(category_summary)

In [ ]:
# TODO：绘制并保存柱状图
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

# 绘制用户数柱状图
bars = ax_bar.bar(category_summary[category_field].astype(str), category_summary["用户数"], color="#4C72B0", alpha=0.7)

# 添加数值标签
for bar in bars:
    height = bar.get_height()
    ax_bar.text(bar.get_x() + bar.get_width()/2., height,
               f"{int(height)}",
               ha='center', va='bottom')

# 设置标题和标签
ax_bar.set_title("不同城市等级的用户数量分布", fontsize=14, fontweight="bold")
ax_bar.set_xlabel("城市等级 (CityTier)", fontsize=12)
ax_bar.set_ylabel("用户数量", fontsize=12)
ax_bar.grid(axis='y', alpha=0.3)

bar_path = OUTPUT_DIR / "01_category_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()

assert bar_path.exists() and bar_path.stat().st_size > 0, "柱状图尚未保存"
print("已输出：", bar_path.relative_to(ROOT))

### 柱状图结论

- 观察：请填写。
- 证据：请填写具体数值、差异和样本量。
- 边界：请填写该图不能证明什么。


## 任务2：用户行为散点图

要求：选择两个数值字段，一行代表一个用户，颜色区分`Churn`，设置透明度。


In [ ]:
# TODO：选择两个数值字段，例如OrderCount与CashbackAmount
x_field = "OrderCount"
y_field = "CashbackAmount"

assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])

fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

# 按Churn分组绘制散点图
churn_0 = df[df["Churn"] == 0]
churn_1 = df[df["Churn"] == 1]

ax_scatter.scatter(churn_0[x_field], churn_0[y_field], 
                  alpha=0.5, label="留存用户", color="#55A868", s=30)
ax_scatter.scatter(churn_1[x_field], churn_1[y_field], 
                  alpha=0.6, label="流失用户", color="#C44E52", s=30)

# 设置标题和标签
ax_scatter.set_title("订单数量与返现金额散点图（按流失状态分组）", fontsize=14, fontweight="bold")
ax_scatter.set_xlabel(f"{x_field} (订单数量)", fontsize=12)
ax_scatter.set_ylabel(f"{y_field} (返现金额)", fontsize=12)
ax_scatter.legend(fontsize=11)
ax_scatter.grid(alpha=0.3)

scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()

assert scatter_path.exists() and scatter_path.stat().st_size > 0, "散点图尚未保存"
print("已输出：", scatter_path.relative_to(ROOT))

### 散点图结论

- 观察：请填写。
- 证据：请填写两个变量的关系、聚集或异常。
- 边界：相关关系不等于因果关系。


## 任务3：有序阶段折线图

当前数据没有日期。建议使用`TenureGroup`或`SatisfactionScore`，并明确写成“阶段比较”。


In [ ]:
TENURE_ORDER = ["新用户", "0-6个月", "7-12个月", "13-24个月", "24个月以上"]

# TODO：准备有序绘图数据
ordered_field = "TenureGroup"
ordered_summary = (
    df.groupby(ordered_field, observed=True)
      .agg(用户数=("CustomerID", "nunique"), 流失率=("Churn", "mean"))
      .reset_index()
)

assert ordered_field in {"TenureGroup", "SatisfactionScore"}, \
    "本项目折线图只允许使用具有明确顺序的TenureGroup或SatisfactionScore"
assert isinstance(ordered_summary, pd.DataFrame)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)
display(ordered_summary)

In [ ]:
# TODO：绘制折线图；若绘制流失率，应标注比例和样本量
fig_line, ax_line = plt.subplots(figsize=(10, 6))

# 绘制流失率折线
x_values = range(len(ordered_summary))
ax_line.plot(x_values, ordered_summary["流失率"], 
            marker='o', linewidth=2, markersize=8, color="#C44E52")

# 标注每个点的流失率和样本量
for idx, row in ordered_summary.iterrows():
    ax_line.annotate(f"{row['流失率']:.1%}\n(n={int(row['用户数'])})", 
                    (idx, row["流失率"]),
                    textcoords="offset points", xytext=(0, 10),
                    ha='center', fontsize=9)

# 设置标题和标签
ax_line.set_title("用户使用时长与流失率关系", fontsize=14, fontweight="bold")
ax_line.set_xlabel("使用时长分组", fontsize=12)
ax_line.set_ylabel("流失率", fontsize=12)
ax_line.set_xticks(x_values)
ax_line.set_xticklabels(ordered_summary[ordered_field])
ax_line.grid(alpha=0.3)
ax_line.set_ylim(0, max(ordered_summary["流失率"]) * 1.3)

line_path = OUTPUT_DIR / "03_ordered_line.png"
fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.show()

assert line_path.exists() and line_path.stat().st_size > 0, "折线图尚未保存"
print("已输出：", line_path.relative_to(ROOT))

### 折线图结论

- 观察：请填写。
- 证据：请填写具体数值和样本量。
- 边界：这是有序阶段比较，不是月度、年度或历史时间趋势。


## 任务4：整体构成图

类别少于或等于5个时可以使用饼图或环形图；否则改用柱状图。必须在选择理由中说明判断。


In [ ]:
# TODO：选择构成字段并准备汇总表
composition_field = "PreferredPaymentMode"
composition_summary = df.groupby(composition_field).agg(
    用户数=("CustomerID", "nunique")
).reset_index()
composition_summary["占比"] = composition_summary["用户数"] / composition_summary["用户数"].sum()
composition_summary = composition_summary.sort_values("占比", ascending=False).reset_index(drop=True)

assert composition_field in df.columns
assert isinstance(composition_summary, pd.DataFrame)
assert {composition_field, "用户数", "占比"}.issubset(composition_summary.columns)
assert np.isclose(composition_summary["占比"].sum(), 1.0), "构成占比之和应为1"
display(composition_summary)

In [ ]:
# TODO：类别不超过5个时绘制环形图，否则绘制柱状图
fig_composition, ax_composition = plt.subplots(figsize=(10, 6))

n_categories = len(composition_summary)

if n_categories <= 5:
    # 绘制环形图
    wedges, texts, autotexts = ax_composition.pie(
        composition_summary["占比"],
        labels=composition_summary[composition_field],
        autopct='%1.1f%%',
        startangle=90,
        pctdistance=0.85,
        colors=["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
    )
    # 添加中心圆形成环形效果
    centre_circle = plt.Circle((0,0), 0.70, fc='white')
    fig_composition.gca().add_artist(centre_circle)
    ax_composition.set_title("用户首选支付方式分布", fontsize=14, fontweight="bold")
else:
    # 绘制柱状图
    bars = ax_composition.bar(composition_summary[composition_field], 
                              composition_summary["占比"] * 100, color="#4C72B0", alpha=0.7)
    ax_composition.set_title("用户首选支付方式分布", fontsize=14, fontweight="bold")
    ax_composition.set_ylabel("占比 (%)", fontsize=12)
    ax_composition.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')

ax_composition.axis('equal')  # 保证饼图为圆形

composition_path = OUTPUT_DIR / "04_composition_chart.png"
fig_composition.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.show()

assert composition_path.exists() and composition_path.stat().st_size > 0, "构成图尚未保存"
print("已输出：", composition_path.relative_to(ROOT))

### 构成图结论

- 观察：请填写。
- 证据：请填写主要类别占比。
- 边界：请说明该图适合或不适合进行哪些比较。


## 检查点2与3：基础图表、优化和解释

逐项使用`docs/day06_chart_checklist.md`检查。确认比率图给出样本量、中文正常、颜色含义一致。


In [ ]:
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5_000, f"图片可能为空或质量过低：{path.name}"

print("检查点2通过：4张独立图已生成")
print("检查点3需要结合图表和文字结论人工复核")


## 任务5：2×2综合图

重新在4个子图中绘制核心内容，不要把4张PNG作为截图拼接。统一标题、颜色、字体和留白。


In [ ]:
fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

# TODO：分别在axes[0,0]、axes[0,1]、axes[1,0]、axes[1,1]绘制4张核心图

# 子图1: 城市等级用户数柱状图
ax1 = axes[0, 0]
ax1.bar(category_summary[category_field].astype(str), category_summary["用户数"], color="#4C72B0", alpha=0.7)
ax1.set_title("城市等级用户分布", fontsize=12, fontweight="bold")
ax1.set_ylabel("用户数量")
ax1.grid(axis='y', alpha=0.3)

# 子图2: 订单数与返现金额散点图
ax2 = axes[0, 1]
churn_0 = df[df["Churn"] == 0]
churn_1 = df[df["Churn"] == 1]
ax2.scatter(churn_0[x_field], churn_0[y_field], alpha=0.5, label="留存", color="#55A868", s=20)
ax2.scatter(churn_1[x_field], churn_1[y_field], alpha=0.6, label="流失", color="#C44E52", s=20)
ax2.set_title("订单数 vs 返现金额", fontsize=12, fontweight="bold")
ax2.set_xlabel("订单数量")
ax2.set_ylabel("返现金额")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# 子图3: 使用时长流失率折线图
ax3 = axes[1, 0]
x_vals = range(len(ordered_summary))
ax3.plot(x_vals, ordered_summary["流失率"], marker='o', linewidth=2, color="#C44E52")
ax3.set_title("使用时长与流失率", fontsize=12, fontweight="bold")
ax3.set_xticks(x_vals)
ax3.set_xticklabels(ordered_summary[ordered_field], fontsize=9)
ax3.set_ylabel("流失率")
ax3.grid(alpha=0.3)

# 子图4: 支付方式构成环形图
ax4 = axes[1, 1]
ax4.pie(composition_summary["占比"], labels=composition_summary[composition_field],
        autopct='%1.0f%%', startangle=90, pctdistance=0.8, textprops={'fontsize': 8})
centre_circle = plt.Circle((0,0), 0.65, fc='white')
ax4.add_artist(centre_circle)
ax4.set_title("支付方式分布", fontsize=12, fontweight="bold")
ax4.axis('equal')

fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])

summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()

assert summary_path.exists() and summary_path.stat().st_size > 0, "综合图尚未保存"
print("已输出：", summary_path.relative_to(ROOT))

## 综合发现与局限

1. 综合发现1：请填写，并给出证据。
2. 综合发现2：请填写，并给出证据。
3. 综合发现3：请填写，并给出证据。
4. 数据或方法局限：请填写。

注意：`CashbackAmount`是返现金额，不是销售额、收入或GMV。


## 任务6：图表清单与检查点4

清单是第7天Flask读取图表说明的基础。每张图填写业务问题、图表类型、主要发现和局限。


In [ ]:
# TODO：填写5行清单，不得保留“请填写”
chart_manifest = pd.DataFrame([
    {"chart_id": "01", "file_name": "01_category_bar.png", "business_question": "不同城市等级的用户数量分布如何？", "chart_type": "bar", "key_finding": "一线城市用户数量最多，三线城市用户数量最少", "limitation": "仅展示用户数分布，未结合流失率和消费行为综合分析"},
    {"chart_id": "02", "file_name": "02_behavior_scatter.png", "business_question": "订单数量与返现金额的关系及流失用户特征？", "chart_type": "scatter", "key_finding": "返现金额较高的用户中流失比例相对较高，订单数集中在1-4单区间", "limitation": "散点重叠较多，难以精确判断密度分布，需结合热力图补充"},
    {"chart_id": "03", "file_name": "03_ordered_line.png", "business_question": "用户使用时长与流失率的变化趋势？", "chart_type": "line", "key_finding": "新用户流失率最高，随着使用时长增加流失率整体呈下降趋势", "limitation": "时长分组样本量不均，部分组样本量较小可能影响结论稳定性"},
    {"chart_id": "04", "file_name": "04_composition_chart.png", "business_question": "用户首选支付方式的构成比例？", "chart_type": "pie_or_bar", "key_finding": "借记卡是最主流的支付方式，其次是信用卡，货到付款占比最低", "limitation": "仅展示首选方式分布，未分析支付方式与流失率的关联关系"},
    {"chart_id": "05", "file_name": "day06_visualization_summary.png", "business_question": "整体概览", "chart_type": "dashboard", "key_finding": "多维度综合呈现用户分布、行为、时长和支付特征，形成完整分析视图", "limitation": "综合图信息密度高，单图细节展示有限，需配合单图深入解读"},
])

assert len(chart_manifest) == 5
assert not chart_manifest.astype(str).apply(lambda col: col.str.contains("请填写").any()).any(), \
    "请完成图表清单"

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")
display(chart_manifest)

In [ ]:
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]
missing_outputs = [str(path.relative_to(ROOT)) for path in required_outputs if not path.exists()]
assert not missing_outputs, f"缺少成果文件：{missing_outputs}"

manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert set(manifest_check["file_name"]) == {path.name for path in required_outputs[:-1]}

print("检查点4通过：第6天成果物完整")
print("下一步：重启内核并从头运行，然后执行提交检查脚本并推送GitHub。")
